# Claude로 Wikipedia를 반복 검색하기

[면책 사항: 이 notebook은 Claude 2 모델로 작성되었으며 legacy로 간주됩니다.]

일부 질문은 Claude가 알고 있는 지식만으로 답할 수 없습니다. 최신 사건에 관한 질문일 수도 있습니다. Claude가 답을 외우지 못한 아주 자세한 질문일 수도 있습니다. 걱정하지 마세요! Prompting과 scaffolding을 제공하면 Claude는 웹을 검색하여 답을 찾을 수 있습니다. 이 notebook에서는 질문의 답을 찾기 위해 Wikipedia를 검색할 수 있는 가상 연구 assistant를 만들겠습니다. 같은 접근 방식을 사용하여 Claude가 더 넓은 웹이나 제공한 문서 집합을 검색하도록 할 수 있습니다.

어떤 접근 방식일까요? 넓게 보면 "tool use" 범주에 속합니다. Search tool을 만들고 Claude에 알려준 뒤 작업하도록 합니다. Pseudocode로 나타내면 다음과 같습니다:

1. Search tool의 설명, 가장 잘 활용하는 방법, "호출"하는 방법(특수 문자열 출력)을 Claude의 prompt에 제공합니다.
2. Claude에 질문을 전달합니다.
3. Claude는 평소처럼 token을 생성합니다. 특수 문자열을 생성하면 token 생성 stream을 종료하고 search API에 query를 보냅니다.
4. 1단계의 prompt에 Claude가 search 호출 문자열까지 생성한 모든 내용과 API 호출 결과를 더하여 새 prompt를 구성합니다.
5. Claude가 완료했다고 판단할 때까지 반복합니다.

Tool use 및 retrieval을 위한 prompt를 자세히 살펴보겠습니다.

### Prompt

In [101]:
# Tool 설명 Prompt
wikipedia_prompt = """You will be asked a question by a human user. You have access to the following tool to help answer the question. <tool_description> Search Engine Tool * The search engine will exclusively search over Wikipedia for pages similar to your query. It returns for each page its title and full page content. Use this tool if you want to get up-to-date and comprehensive information on a topic to help answer queries. Queries should be as atomic as possible -- they only need to address one part of the user's question. For example, if the user's query is "what is the color of a basketball?", your search query should be "basketball". Here's another example: if the user's question is "Who created the first neural network?", your first query should be "neural network". As you can see, these queries are quite short. Think keywords, not phrases. * At any time, you can make a call to the search engine using the following syntax: <search_query>query_word</search_query>. * You'll then get results back in <search_result> tags.</tool_description>"""
print(wikipedia_prompt)

You will be asked a question by a human user. You have access to the following tool to help answer the question. <tool_description> Search Engine Tool * The search engine will exclusively search over Wikipedia for pages similar to your query. It returns for each page its title and full page content. Use this tool if you want to get up-to-date and comprehensive information on a topic to help answer queries. Queries should be as atomic as possible -- they only need to address one part of the user's question. For example, if the user's query is "what is the color of a basketball?", your search query should be "basketball". Here's another example: if the user's question is "Who created the first neural network?", your first query should be "neural network". As you can see, these queries are quite short. Think keywords, not phrases. * At any time, you can make a call to the search engine using the following syntax: <search_query>query_word</search_query>. * You'll then get results back in <

이 prompt에는 Wikipedia를 올바르게 검색하는 방법에 대한 조언이 많다는 점에 주목하세요. Google은 query parsing logic이 매우 좋아서, 우리는 아무 말이나 입력하고도 괜찮은 결과를 얻는 데 익숙합니다. Wikipedia 검색은 그렇지 않습니다. 예를 들어 "아랍에미리트에서 감자를 구매하는 가장 좋은 방법은 무엇인가요"라는 query를 생각해 보세요. [Wikipedia에서 이 검색의 상위 결과](https://en.wikipedia.org/w/index.php?search=What%27s+the+best+way+to+purchase+potatoes+in+the+United+Arab+Emirates&title=Special:Search&profile=advanced&fulltext=1&ns0=1)는 미국의 노예제, 1973년 석유 위기, Wendy's, Tim Horton's(??)입니다. 반면 Google은 올바르게 Carrefour UAE로 바로 안내합니다.

또 다른 차이는 Wikipedia 검색이 전체 페이지를 반환한다는 점입니다. Vector search에서는 더 좁은 chunk를 받을 수 있으므로, 결과를 더 많이 요청하거나 더 구체적인 query를 사용하거나 둘 다 할 수 있습니다. 여기서 큰 틀의 교훈은 이러한 선택에 따라 결과가 크게 달라질 수 있으므로 주의해야 한다는 것입니다!

In [103]:
retrieval_prompt = """Before beginning to research the user's question, first think for a moment inside <scratchpad> tags about what information is necessary for a well-informed answer. If the user's question is complex, you may need to decompose the query into multiple subqueries and execute them individually. Sometimes the search engine will return empty search results, or the search results may not contain the information you need. In such cases, feel free to try again with a different query.

After each call to the Search Engine Tool, reflect briefly inside <search_quality></search_quality> tags about whether you now have enough information to answer, or whether more information is needed. If you have all the relevant information, write it in <information></information> tags, WITHOUT actually answering the question. Otherwise, issue a new search.

Here is the user's question: <question>{query}</question> Remind yourself to make short queries in your scratchpad as you plan out your strategy."""
print(retrieval_prompt)

Before beginning to research the user's question, first think for a moment inside <scratchpad> tags about what information is necessary for a well-informed answer. If the user's question is complex, you may need to decompose the query into multiple subqueries and execute them individually. Sometimes the search engine will return empty search results, or the search results may not contain the information you need. In such cases, feel free to try again with a different query. 

After each call to the Search Engine Tool, reflect briefly inside <search_quality></search_quality> tags about whether you now have enough information to answer, or whether more information is needed. If you have all the relevant information, write it in <information></information> tags, WITHOUT actually answering the question. Otherwise, issue a new search.

Here is the user's question: <question>{query}</question> Remind yourself to make short queries in your scratchpad as you plan out your strategy.



여기서는 일반적인 chain-of-thought의 이유로 scratchpad를 사용합니다. Claude가 질문에 답할 일관된 계획을 세우게 합니다. 검색 품질에 대한 성찰은 Claude가 끈기 있게 검색하고 관련 정보를 모두 수집하기 전에 성급하게 질문에 답하지 않도록 유도하는 데 사용합니다. 그런데 왜 Claude에 정보를 종합하고 바로 답하지 말라고 하는 것일까요?

In [104]:
answer_prompt = "Here is a user query: <query>{query}</query>. Here is some relevant information: <information>{information}</information>. Please answer the question using the relevant information."
print(answer_prompt)

Here is a user query: <query>{query}</query>. Here is some relevant information: <information>{information}</information>. Please answer the question using the relevant information.


정보를 추출하여 새 query로 Claude에 제공하면 Claude가 정보를 올바른 답으로 종합하는 데 모든 주의를 집중할 수 있게 됩니다. 이 단계가 없으면 Claude가 결과에 따라 답을 정하기보다, 때때로 답을 미리 정하고 검색 결과로 그 답을 "정당화"하는 것을 발견했습니다.

이제 검색 + retrieval + reprompting의 pseudocode를 구현하는 여러 코드가 이어집니다.

### 검색 구현

In [88]:
import re
from abc import abstractmethod
from dataclasses import dataclass

import wikipedia
from anthropic import AI_PROMPT, HUMAN_PROMPT, Anthropic


@dataclass
class SearchResult:
    """
    단일 검색 결과.
    """

    content: str


class SearchTool:
    """
    Query를 실행하고 형식을 정리한 검색 결과 문자열을 반환할 수 있는 search tool.
    """

    def __init__():
        pass

    @abstractmethod
    def raw_search(self, query: str, n_search_results_to_use: int) -> list[SearchResult]:
        """
        Searcher로 query를 실행한 뒤 형식을 정리하지 않은 원시 검색 결과를 반환합니다.

        :param query: 실행할 query.
        :param n_search_results_to_use: 반환할 결과 수.
        """
        raise NotImplementedError()

    @abstractmethod
    def process_raw_search_results(
        self,
        results: list[SearchResult],
    ) -> list[str]:
        """
        검색 결과에서 원시 검색 내용을 추출하여 Claude에 전달할 수 있는 문자열 목록을 반환합니다.

        :param results: 추출할 검색 결과.
        """
        raise NotImplementedError()

    def search_results_to_string(self, extracted: list[str]) -> str:
        """
        추출한 검색 결과를 결합하고 문자열로 형식을 정리합니다.

        :param extracted: 형식을 정리할 추출된 검색 결과.
        """
        result = "\n".join(
            [
                f'<item index="{i + 1}">\n<page_content>\n{r}\n</page_content>\n</item>'
                for i, r in enumerate(extracted)
            ]
        )
        return result

    def wrap_search_results(self, extracted: list[str]) -> str:
        """
        <search_results> tag를 포함하여 추출한 검색 결과를 문자열로 정리합니다.

        :param extracted: 형식을 정리할 추출된 검색 결과.
        """
        return f"\n<search_results>\n{self.search_results_to_string(extracted)}\n</search_results>"

    def search(self, query: str, n_search_results_to_use: int) -> str:
        raw_search_results = self.raw_search(query, n_search_results_to_use)
        processed_search_results = self.process_raw_search_results(raw_search_results)
        displayable_search_results = self.wrap_search_results(processed_search_results)
        return displayable_search_results

In [ ]:
@dataclass
class WikipediaSearchResult(SearchResult):
    title: str


class WikipediaSearchTool(SearchTool):
    def __init__(self, truncate_to_n_tokens: int | None = 5000):
        self.truncate_to_n_tokens = truncate_to_n_tokens
        if truncate_to_n_tokens is not None:
            self.tokenizer = Anthropic().get_tokenizer()

    def raw_search(self, query: str, n_search_results_to_use: int) -> list[WikipediaSearchResult]:
        search_results = self._search(query, n_search_results_to_use)
        return search_results

    def process_raw_search_results(self, results: list[WikipediaSearchResult]) -> list[str]:
        processed_search_results = [
            f"Page Title: {result.title.strip()}\nPage Content:\n{self.truncate_page_content(result.content)}"
            for result in results
        ]
        return processed_search_results

    def truncate_page_content(self, page_content: str) -> str:
        if self.truncate_to_n_tokens is None:
            return page_content.strip()
        else:
            return self.tokenizer.decode(
                self.tokenizer.encode(page_content).ids[: self.truncate_to_n_tokens]
            ).strip()

    def _search(self, query: str, n_search_results_to_use: int) -> list[WikipediaSearchResult]:
        results: list[str] = wikipedia.search(query)
        search_results: list[WikipediaSearchResult] = []
        for result in results:
            if len(search_results) >= n_search_results_to_use:
                break
            try:
                page = wikipedia.page(result)
                print(page.url)
            except wikipedia.exceptions.WikipediaException:
                # Wikipedia API는 약간 불안정하므로 로드에 실패한 페이지는 건너뜁니다
                continue
            content = page.content
            title = page.title
            search_results.append(WikipediaSearchResult(content=content, title=title))
        return search_results

In [100]:
def extract_between_tags(tag: str, string: str, strip: bool = True) -> list[str]:
    ext_list = re.findall(rf"<{tag}\s?>(.+?)</{tag}\s?>", string, re.DOTALL)
    if strip:
        ext_list = [e.strip() for e in ext_list]
    return ext_list


class ClientWithRetrieval(Anthropic):
    def __init__(self, search_tool: SearchTool, verbose: bool = True, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.search_tool = search_tool
        self.verbose = verbose

    # 보조 method
    def _search_query_stop(
        self, partial_completion: str, n_search_results_to_use: int
    ) -> tuple[list[SearchResult], str]:
        search_query = extract_between_tags("search_query", partial_completion + "</search_query>")
        if search_query is None:
            raise Exception(
                "Completion with retrieval failed as partial completion returned mismatched <search_query> tags."
            )
        print(f"Running search query against SearchTool: {search_query}")
        search_results = self.search_tool.raw_search(search_query, n_search_results_to_use)
        extracted_search_results = self.search_tool.process_raw_search_results(search_results)
        formatted_search_results = self.search_tool.wrap_search_results(extracted_search_results)
        return search_results, formatted_search_results

    def retrieve(
        self,
        query: str,
        model: str,
        n_search_results_to_use: int = 3,
        stop_sequences: list[str] = None,
        max_tokens_to_sample: int = 1000,
        max_searches_to_try: int = 5,
        temperature: float = 1.0,
    ) -> tuple[list[SearchResult], str]:
        if stop_sequences is None:
            stop_sequences = [HUMAN_PROMPT]
        prompt = (
            f"{HUMAN_PROMPT} {wikipedia_prompt} {retrieval_prompt.format(query=query)}{AI_PROMPT}"
        )
        starting_prompt = prompt
        print("Starting prompt:", starting_prompt)
        token_budget = max_tokens_to_sample
        all_raw_search_results: list[SearchResult] = []
        for tries in range(max_searches_to_try):
            partial_completion = self.completions.create(
                prompt=prompt,
                stop_sequences=stop_sequences + ["</search_query>"],
                model=model,
                max_tokens_to_sample=token_budget,
                temperature=temperature,
            )
            partial_completion, stop_reason, stop_seq = (
                partial_completion.completion,
                partial_completion.stop_reason,
                partial_completion.stop,
            )
            print(partial_completion)
            token_budget -= self.count_tokens(partial_completion)
            prompt += partial_completion
            if stop_reason == "stop_sequence" and stop_seq == "</search_query>":
                print(f"Attempting search number {tries}.")
                raw_search_results, formatted_search_results = self._search_query_stop(
                    partial_completion, n_search_results_to_use
                )
                prompt += "</search_query>" + formatted_search_results
                all_raw_search_results += raw_search_results
            else:
                break
        final_model_response = prompt[len(starting_prompt) :]
        return all_raw_search_results, final_model_response

    # 주요 method
    def completion_with_retrieval(
        self,
        query: str,
        model: str,
        n_search_results_to_use: int = 3,
        stop_sequences: list[str] = None,
        max_tokens_to_sample: int = 1000,
        max_searches_to_try: int = 5,
        temperature: float = 1.0,
    ) -> str:
        if stop_sequences is None:
            stop_sequences = [HUMAN_PROMPT]
        _, retrieval_response = self.retrieve(
            query,
            model=model,
            n_search_results_to_use=n_search_results_to_use,
            stop_sequences=stop_sequences,
            max_tokens_to_sample=max_tokens_to_sample,
            max_searches_to_try=max_searches_to_try,
            temperature=temperature,
        )
        information = extract_between_tags("information", retrieval_response)[-1]
        prompt = f"{HUMAN_PROMPT} {answer_prompt.format(query=query, information=information)}{AI_PROMPT}"
        print("Summarizing:\n", prompt)
        answer = self.completions.create(
            prompt=prompt, model=model, temperature=temperature, max_tokens_to_sample=1000
        ).completion
        return answer

### Query 실행

Query를 실행할 준비가 되었습니다! 다음 특성의 질문을 골라 보겠습니다:
- Claude의 학습 데이터에 있을 가능성이 낮은 최신 질문, 그리고
- 여러 번 검색해야 하는 복합적/복잡한 질문.

In [98]:
import os

# Searcher 생성
wikipedia_search_tool = WikipediaSearchTool()
ANTHROPIC_SEARCH_MODEL = "claude-2"

client = ClientWithRetrieval(
    api_key=os.environ["ANTHROPIC_API_KEY"], verbose=True, search_tool=wikipedia_search_tool
)

query = "Which movie came out first: Oppenheimer, or Are You There God It's Me Margaret?"

augmented_response = client.completion_with_retrieval(
    query=query,
    model=ANTHROPIC_SEARCH_MODEL,
    n_search_results_to_use=1,
    max_searches_to_try=5,
    max_tokens_to_sample=1000,
    temperature=0,
)
print(augmented_response)

Starting prompt: 

Human: You will be asked a question by a human user. You have access to the following tool to help answer the question. <tool_description> Search Engine Tool * The search engine will exclusively search over Wikipedia for pages similar to your query. It returns for each page its title and full page content. Use this tool if you want to get up-to-date and comprehensive information on a topic to help answer queries. Queries should be as atomic as possible -- they only need to address one part of the user's question. For example, if the user's query is "what is the color of a basketball?", your search query should be "basketball". Here's another example: if the user's question is "Who created the first neural network?", your first query should be "neural network". As you can see, these queries are quite short. Think keywords, not phrases. * At any time, you can make a call to the search engine using the following syntax: <search_query>query_word</search_query>. * You'll 

좋습니다. Claude는 계획을 세우고 query를 실행한 뒤 정보를 종합하여 정확한 답을 만들 수 있었습니다. 참고: 추가 정보 추출 단계가 없으면 Claude가 때때로 영화의 개봉일은 정확히 알아내지만 최종 답에서 순서를 잘못 정했습니다. 다른 질문을 해 보겠습니다.

In [99]:
augmented_response = client.completion_with_retrieval(
    query="Who won the 2023 NBA championship? Who was that team's best player in the year 2009?",
    model=ANTHROPIC_SEARCH_MODEL,
    n_search_results_to_use=1,
    max_searches_to_try=5,
    max_tokens_to_sample=1000,
    temperature=0,
)
print(augmented_response)

Starting prompt: 

Human: You will be asked a question by a human user. You have access to the following tool to help answer the question. <tool_description> Search Engine Tool * The search engine will exclusively search over Wikipedia for pages similar to your query. It returns for each page its title and full page content. Use this tool if you want to get up-to-date and comprehensive information on a topic to help answer queries. Queries should be as atomic as possible -- they only need to address one part of the user's question. For example, if the user's query is "what is the color of a basketball?", your search query should be "basketball". Here's another example: if the user's question is "Who created the first neural network?", your first query should be "neural network". As you can see, these queries are quite short. Think keywords, not phrases. * At any time, you can make a call to the search engine using the following syntax: <search_query>query_word</search_query>. * You'll 

이렇게 완성되었습니다! Search tool 코드가 잘 추상화되어 있어 조금만 수정하면 원하는 search API를 사용하도록 바꿀 수 있다는 점을 알 수 있습니다. Tool을 잘 사용하기 위해 필요한 팁을 Claude에 설명하는 것만 기억하세요. 이상적인 query 계획 및 query 구조의 few-shot 예제를 Claude에 제공하여 성능을 더 개선할 수도 있습니다.